# 📚 Lesson Summarizer — Improved Version
**Model:** `facebook/bart-large-cnn`

### ✅ Improvements in this version:
| Area | What changed |
|------|--------------|
| 🎯 Summary Quality | Better generation params + post-processing |
| 📊 Evaluation | ROUGE + BERTScore + METEOR |
| 🧹 Code Structure | Config class + error handling + modular design |

## 1. Install Dependencies

In [1]:
!pip install transformers torch nltk rouge-score bert-score pycocoevalcap

## 2. Imports & NLTK Downloads

In [2]:
import re
import nltk
import warnings
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from bert_score import score as bert_score

warnings.filterwarnings('ignore')

for pkg in ['punkt', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

print("✅ All imports successful.")

✅ All imports successful.


## 3. Configuration Class

In [9]:
@dataclass
class SummarizerConfig:
    # Model
    model_name: str   = "facebook/bart-large-cnn"

    # Chunking
    chunk_size: int   = 250          # ⬇ smaller = less info lost per chunk

    # Generation  (🎯 improved params)
    max_input:  int   = 1024
    max_output: int   = 350          # ⬆ was 120 — allows fuller sentences
    min_output: int   = 50           # ⬆ was 40  — avoids too-short summaries
    num_beams:  int   = 6            # ⬆ was 4   — better beam search
    length_penalty: float = 1.5      # ✨ new — rewards longer, complete sentences
    no_repeat_ngram_size: int = 3    # ✨ new — prevents repeated phrases

# Create one global config
config = SummarizerConfig()
print(config)

SummarizerConfig(model_name='facebook/bart-large-cnn', chunk_size=250, max_input=1024, max_output=350, min_output=50, num_beams=6, length_penalty=1.5, no_repeat_ngram_size=3)


## 4. Load Model & Tokenizer

In [10]:
def load_model(cfg: SummarizerConfig):
    """Load model and tokenizer with error handling."""
    try:
        print(f"Loading model: {cfg.model_name} ...")
        tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
        model     = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name)
        print("✅ Model loaded successfully.")
        return model, tokenizer
    except Exception as e:
        raise RuntimeError(f"❌ Failed to load model: {e}")

model, tokenizer = load_model(config)

Loading model: facebook/bart-large-cnn ...


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

✅ Model loaded successfully.


## 5. Text Processing Functions

In [6]:
def clean_text(text: str) -> str:
    """Normalize whitespace and remove unsupported characters."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text must be a non-empty string.")
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,!?\' ]', '', text)
    return text.strip()


def split_text(text: str, chunk_size: int = 400) -> list[str]:
    """
    Split by sentences first (respects boundaries), then batch
    sentences into chunks of ~chunk_size words.
    🎯 Better than splitting mid-sentence on word count alone.
    """
    sentences = nltk.sent_tokenize(text)
    chunks, current_chunk, current_len = [], [], 0

    for sentence in sentences:
        word_count = len(sentence.split())
        if current_len + word_count > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk, current_len = [], 0
        current_chunk.append(sentence)
        current_len += word_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def postprocess_summary(text: str) -> str:
    """
    ✨ New — clean up generated summaries:
    - Remove duplicate sentences
    - Fix spacing around punctuation
    - Capitalize first letter
    """
    sentences = nltk.sent_tokenize(text)
    seen, unique = set(), []
    for s in sentences:
        normalized = s.lower().strip()
        if normalized not in seen:
            seen.add(normalized)
            unique.append(s.strip())

    result = ' '.join(unique)
    result = re.sub(r'\s([?.!,])', r'\1', result)  # fix space before punctuation
    return result[0].upper() + result[1:] if result else result


print("✅ Text processing functions ready.")

✅ Text processing functions ready.


## 6. Summarization Pipeline

In [7]:
def summarize_chunks(chunks: list[str], model, tokenizer, cfg: SummarizerConfig) -> list[str]:
    """Summarize each chunk using improved generation parameters."""
    summaries = []

    for idx, chunk in enumerate(chunks, start=1):
        print(f"  Summarizing chunk {idx}/{len(chunks)}...")
        try:
            inputs = tokenizer(
                chunk,
                return_tensors="pt",
                max_length=cfg.max_input,
                truncation=True,
            )
            summary_ids = model.generate(
                inputs["input_ids"],
                num_beams=cfg.num_beams,
                max_length=cfg.max_output,
                min_length=cfg.min_output,
                length_penalty=cfg.length_penalty,
                no_repeat_ngram_size=cfg.no_repeat_ngram_size,
                early_stopping=True,
            )
            text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            summaries.append(text)
        except Exception as e:
            print(f"  ⚠️ Skipping chunk {idx} due to error: {e}")

    return summaries


def summarize_lesson(lesson_text: str, cfg: SummarizerConfig = config) -> str:
    """
    Full pipeline:
      Clean → Split (sentence-aware) → Summarize → Combine → Postprocess
    """
    print("Step 1: Cleaning text...")
    cleaned = clean_text(lesson_text)

    print("Step 2: Splitting into sentence-aware chunks...")
    chunks = split_text(cleaned, cfg.chunk_size)
    print(f"  → {len(chunks)} chunk(s) created.")

    print("Step 3: Summarizing...")
    summaries = summarize_chunks(chunks, model, tokenizer, cfg)

    print("Step 4: Combining & postprocessing...")
    combined = " ".join(summaries)
    final    = postprocess_summary(combined)

    return final


print("✅ Summarization pipeline ready.")

✅ Summarization pipeline ready.


## 7. Run on Lesson Text

In [11]:
LESSON_TEXT = """
Artificial Intelligence (AI) is transforming the education sector in many ways,
changing how students learn and how teachers teach. AI technologies are being
integrated into educational platforms to provide personalized learning experiences,
automate administrative tasks, and improve student engagement. The use of AI in
education is expected to grow significantly in the coming years as technology
continues to evolve.One of the most important applications of artificial intelligence in education is
personalized learning. Traditional education systems often follow a one-size-fits-all
approach, where all students are taught the same material at the same pace. However,
students have different learning styles, abilities, and interests. AI systems can
analyze student performance, learning speed, and behavior to create personalized
learning paths for each student. This helps students learn more effectively and
improves their academic performance.Another major application of AI in education is intelligent tutoring systems. These
systems act as virtual tutors that can help students understand difficult concepts,
answer questions, and provide explanations. Intelligent tutoring systems use natural
language processing and machine learning algorithms to interact with students in a
conversational way. This allows students to receive immediate feedback and support
without waiting for a teacher.AI is also used for automated grading and assessment. Teachers often spend a lot of
time grading assignments, quizzes, and exams. AI systems can automatically grade
multiple-choice questions, short answers, and even essays in some cases. This reduces
the workload on teachers and allows them to focus more on teaching and interacting
with students. Automated grading systems can also provide detailed feedback to
students, helping them understand their mistakes and improve their performance.Another important use of AI in education is content generation and summarization. AI
systems can generate study materials, create quizzes, summarize lessons, and extract
important keywords from educational content. This is especially useful for online
learning platforms where students need quick summaries and review materials.
Summarization systems help students review lessons faster and focus on the most
important information.AI can also be used to analyze student behavior and predict student performance. By
analyzing attendance records, assignment submissions, quiz scores, and participation,
AI systems can identify students who are at risk of failing or dropping out. Teachers
and administrators can then provide additional support to these students before it is
too late. This predictive analysis can improve student retention and success rates.Despite the many advantages of AI in education, there are also some challenges. One
of the main challenges is data privacy and security. AI systems require large amounts
of student data to function effectively, and this data must be protected from
unauthorized access. Another challenge is the cost of implementing AI systems in
schools and universities. Some educational institutions may not have the resources to
adopt advanced AI technologies.There is also a concern that too much reliance on AI could reduce human interaction
in education. Teachers play an important role not only in teaching academic subjects
but also in developing students social skills, critical thinking, and creativity.
Therefore, AI should be used as a tool to support teachers, not replace them.In the future, artificial intelligence is expected to play an even bigger role in
education. We may see fully intelligent learning platforms that can teach students,
evaluate their performance, and adapt the curriculum automatically based on student
progress. Virtual reality and AI may also be combined to create immersive learning
environments where students can learn through interactive simulations.In conclusion, artificial intelligence has the potential to revolutionize education
by providing personalized learning, intelligent tutoring, automated grading, content
summarization, and predictive analytics. However, it is important to address
challenges such as data privacy, cost, and maintaining human interaction in education.
If implemented correctly, AI can greatly improve the quality of education and make
learning more accessible and effective for students around the world.
"""

generated_summary = summarize_lesson(LESSON_TEXT, config)

print("\n" + "="*60)
print("GENERATED SUMMARY")
print("="*60)
print(generated_summary)
print(f"\n📏 Summary length: {len(generated_summary.split())} words")

Step 1: Cleaning text...
Step 2: Splitting into sentence-aware chunks...
  → 3 chunk(s) created.
Step 3: Summarizing...
  Summarizing chunk 1/3...
  Summarizing chunk 2/3...
  Summarizing chunk 3/3...
Step 4: Combining & postprocessing...

GENERATED SUMMARY
AI technologies are being integrated into educational platforms to provide personalized learning experiences, automate administrative tasks, and improve student engagement. Traditional education systems often follow a onesizefitsall approach. AI systems can analyze student performance, learning speed, and behavior to create personalized learning paths for each student. AI systems can generate study materials, create quizzes, summarize lessons, and extract important keywords from educational content. AI can also be used to analyze student behavior and predict student performance. Despite the many advantages of AI in education, there are also some challenges. In the future, artificial intelligence is expected to play an even bigger ro

In [12]:
def evaluate_summary(reference: str, generated: str) -> dict:
    """
    Evaluate generated summary using:
      - ROUGE-1, ROUGE-2, ROUGE-L  (lexical overlap)
      - BERTScore                  (semantic similarity)
      - METEOR                     (synonym-aware overlap)
    """
    results = {}

    # ── ROUGE ──────────────────────────────────────────────
    print("Computing ROUGE scores...")
    scorer  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r_scores = scorer.score(reference, generated)
    results['rouge'] = {
        k: {'precision': round(v.precision, 4),
            'recall':    round(v.recall,    4),
            'f1':        round(v.fmeasure,  4)}
        for k, v in r_scores.items()
    }

    # ── BERTScore ─────────────────────────────────────────
    print("Computing BERTScore (semantic similarity)...")
    P, R, F1 = bert_score([generated], [reference], lang='en', verbose=False)
    results['bertscore'] = {
        'precision': round(P.mean().item(), 4),
        'recall':    round(R.mean().item(), 4),
        'f1':        round(F1.mean().item(), 4),
    }

    # ── METEOR ────────────────────────────────────────────
    print("Computing METEOR (synonym-aware)...")
    from nltk.translate.meteor_score import meteor_score
    ref_tokens = nltk.word_tokenize(reference.lower())
    gen_tokens = nltk.word_tokenize(generated.lower())
    results['meteor'] = round(meteor_score([ref_tokens], gen_tokens), 4)

    return results


scores = evaluate_summary(LESSON_TEXT, generated_summary)
print("\n✅ Evaluation complete.")

Computing ROUGE scores...
Computing BERTScore (semantic similarity)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Computing METEOR (synonym-aware)...

✅ Evaluation complete.


## 9. Display Results

In [13]:
print("=" * 55)
print(" EVALUATION RESULTS")
print("=" * 55)

# ROUGE table
print(f"\n{'ROUGE':}")
print(f"  {'Metric':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("  " + "-" * 42)
for metric, vals in scores['rouge'].items():
    print(f"  {metric.upper():<12} {vals['precision']:>10.4f} {vals['recall']:>10.4f} {vals['f1']:>10.4f}")

# BERTScore
b = scores['bertscore']
print(f"\n{'BERTScore (semantic)':}")
print(f"  {'Metric':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("  " + "-" * 42)
print(f"  {'BERTScore':<12} {b['precision']:>10.4f} {b['recall']:>10.4f} {b['f1']:>10.4f}")

# METEOR
print(f"\n{'METEOR (synonym-aware)':}")
print(f"  Score: {scores['meteor']:.4f}")

print("\n" + "=" * 55)
print("💡 Interpretation:")
print("  ROUGE    → measures word overlap (0–1, higher = better)")
print("  BERTScore → measures meaning similarity (0–1, higher = better)")
print("  METEOR   → synonym-aware overlap (0–1, higher = better)")
print("=" * 55)

 EVALUATION RESULTS

ROUGE
  Metric        Precision     Recall         F1
  ------------------------------------------
  ROUGE1           0.9929     0.2186     0.3582
  ROUGE2           0.9424     0.2063     0.3385
  ROUGEL           0.9929     0.2186     0.3582

BERTScore (semantic)
  Metric        Precision     Recall         F1
  ------------------------------------------
  BERTScore        0.9056     0.8453     0.8744

METEOR (synonym-aware)
  Score: 0.1836

💡 Interpretation:
  ROUGE    → measures word overlap (0–1, higher = better)
  BERTScore → measures meaning similarity (0–1, higher = better)
  METEOR   → synonym-aware overlap (0–1, higher = better)
